In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install tokenizers

In [ ]:
from tokenizers import ByteLevelBPETokenizer
import os

tokenizer = ByteLevelBPETokenizer()

files = ["/content/drive/MyDrive/Datasets/Text Datasets/expert1_train_final.txt"]

print("Ξεκινάει η εκπαίδευση του Tokenizer...")

tokenizer.train(
    files=files,
    vocab_size=8192,
    min_frequency=2,
    special_tokens=[
        "<s>",
        "<pad>",
        "</s>",
        "<unk>",
        "<mask>",
    ]
)

os.makedirs("agro_tokenizer", exist_ok=True)
tokenizer.save_model("agro_tokenizer")

print("Τέλεια! Τα αρχεία vocab.json και merges.txt αποθηκεύτηκαν στον φάκελο 'agro_tokenizer'.")

Ξεκινάει η εκπαίδευση του Tokenizer...
Τέλεια! Τα αρχεία vocab.json και merges.txt αποθηκεύτηκαν στον φάκελο 'agro_tokenizer'.


In [ ]:
from tokenizers import ByteLevelBPETokenizer


tokenizer = ByteLevelBPETokenizer(
    "agro_tokenizer/vocab.json",
    "agro_tokenizer/merges.txt"
)


test_sentence = "Παραγγελία 500kg λίπασμα NPK για αποθήκη Λάρισας."
encoded = tokenizer.encode(test_sentence)

print("Αρχικό κείμενο:", test_sentence)
print("Tokens (Αριθμοί για το PyTorch):", encoded.ids)
print("Πώς το έκοψε σε λέξεις:", encoded.tokens)
decoded = tokenizer.decode(encoded.ids)
print("Αποκωδικοποίηση (Πίσω σε κείμενο):", decoded)

Αρχικό κείμενο: Παραγγελία 500kg λίπασμα NPK για αποθήκη Λάρισας.
Tokens (Αριθμοί για το PyTorch): [737, 7756, 1474, 1904, 8083, 2014, 433, 3909, 3711, 18]
Πώς το έκοψε σε λέξεις: ['ÎłÎ±Ïģ', 'Î±Î³Î³ÎµÎ»Î¯Î±', 'Ġ500', 'kg', 'ĠÎ»Î¯ÏĢÎ±ÏĥÎ¼Î±', 'ĠNPK', 'ĠÎ³Î¹Î±', 'ĠÎ±ÏĢÎ¿Î¸Î®ÎºÎ·', 'ĠÎĽÎ¬ÏģÎ¹ÏĥÎ±ÏĤ', '.']
Αποκωδικοποίηση (Πίσω σε κείμενο): Παραγγελία 500kg λίπασμα NPK για αποθήκη Λάρισας.


In [ ]:
import os
tokenizer_path = "/content/drive/MyDrive/Models/agro_tokenizer"
os.makedirs(tokenizer_path, exist_ok=True)
tokenizer.save_model(tokenizer_path)

['/content/drive/MyDrive/Models/agro_tokenizer/vocab.json',
 '/content/drive/MyDrive/Models/agro_tokenizer/merges.txt']

In [ ]:
import torch
import torch.nn as nn
import math

class ERPLanguageModel(nn.Module):
    def __init__(self, vocab_size=8192, d_model=512, n_heads=8, n_layers=8, max_seq_len=256):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len


        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)


        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True,
            activation="gelu",
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0)


        x = self.token_embedding(x) * math.sqrt(self.d_model) + self.position_embedding(positions)


        x = self.transformer(x, is_causal=True)

        logits = self.lm_head(x)
        return logits


model = ERPLanguageModel().to(device)
print("Το μοντέλο διορθώθηκε με Pre-LN και Embedding Scaling!")

Το μοντέλο διορθώθηκε με Pre-LN και Embedding Scaling!


/tmp/ipykernel_1103/3150878049.py:25: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


In [ ]:
def count_parameters(model):

    return sum(p.numel() for p in model.parameters() if p.requires_grad)

total_params = count_parameters(model)
print(f"Συνολικές εκπαιδεύσιμες παράμετροι: {total_params:,}")

Συνολικές εκπαιδεύσιμες παράμετροι: 33,746,944


In [ ]:
import torch
from tqdm.auto import tqdm
from torch.amp import autocast, GradScaler


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)


scaler = GradScaler('cuda')
epochs = 3

print("Ξεκινάει το Training!")

for epoch in range(epochs):
    model.train()
    total_loss = 0

    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")

    for x, y in progress_bar:
        x, y = x.to(device), y.to(device)


        optimizer.zero_grad(set_to_none=True)

        with autocast('cuda'):
            logits = model(x)
            if isinstance(logits, tuple):
                logits = logits[0]
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))


        scaler.scale(loss).backward()


        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_loss = total_loss / len(dataloader)
    print(f"Τέλος Epoch {epoch+1} | Μέσο Σφάλμα (Loss): {avg_loss:.4f}\n")

Ξεκινάει το Training με το διορθωμένο Pre-LN μοντέλο!


Epoch 1/3:   0%|          | 0/2392 [00:00<?, ?it/s]

Τέλος Epoch 1 | Μέσο Σφάλμα (Loss): 7.8656



Epoch 2/3:   0%|          | 0/2392 [00:00<?, ?it/s]

Τέλος Epoch 2 | Μέσο Σφάλμα (Loss): 3.9544



Epoch 3/3:   0%|          | 0/2392 [00:00<?, ?it/s]

Τέλος Epoch 3 | Μέσο Σφάλμα (Loss): 3.5787



In [ ]:

for batch_x, batch_y in dataloader:
    print("--- ΔΕΙΓΜΑ BATCH ---")
    print("Input x shape:", batch_x.shape)
    print("Target y shape:", batch_y.shape)


    sample_x = batch_x[0][:15].tolist()
    sample_y = batch_y[0][:15].tolist()

    print("X tokens (πρώτα 15):", sample_x)
    print("Y tokens (πρώτα 15):", sample_y)
    break

--- ΔΕΙΓΜΑ BATCH ---
Input x shape: torch.Size([64, 63])
Target y shape: torch.Size([64, 63])
X tokens (πρώτα 15): [1956, 306, 756, 262, 1434, 327, 2520, 3376, 341, 580, 3137, 417, 2608, 1391, 268]
Y tokens (πρώτα 15): [306, 756, 262, 1434, 327, 2520, 3376, 341, 580, 3137, 417, 2608, 1391, 268, 370]


In [ ]:
import torch
import os


drive_output_dir = "/content/drive/MyDrive/Models"
os.makedirs(drive_output_dir, exist_ok=True)


model_path = os.path.join(drive_output_dir, "erp_transformer_30m.pt")


torch.save({
    'model_state_dict': model.state_dict(),
    'hyperparameters': {
        'vocab_size': 8192,
        'd_model': 512,
        'n_heads': 8,
        'n_layers': 8,
        'max_seq_len': 256
    }
}, model_path)

print(f"Το μοντέλο αποθηκεύτηκε με επιτυχία στο Drive: {model_path}")

Το μοντέλο αποθηκεύτηκε με επιτυχία στο Drive: /content/drive/MyDrive/Models/erp_transformer_30m.pt
